# **EXERCISE 1: Linear Regression Analysis -- Objective**

To determine if there is a positive, negative, or any relationship between 
the percent of population in poverty in each county of the United States (independent variable) and 
the median household income in each county of the United States (dependent variable).

# **United States Counties Dataset**

**Description**
Data for 3142 counties in the United States.

**Usage**
county

**Format**
A data frame with 3142 observations on the following 14 variables.

**name**
County names.

**state**
State names.

**pop2000**
Population in 2000.

**pop2010**
Population in 2010.

**pop2017**
Population in 2017.

**pop_change**
Population change from 2010 to 2017.

**poverty**
Percent of population in poverty in 2017.

**homeownership**
Home ownership rate, 2006-2010.

**multi_unit**
Percent of housing units in multi-unit structures, 2006-2010.

**unemployment_rate**
Unemployment rate in 2017.

**metro**
Whether the county contains a metropolitan area.

**median_edu**
Median education level (2013-2017).

**per_capita_income**
Per capita (per person) income (2013-2017).

**median_hh_income**
Median household income.

**smoking_ban**
Describes whether the type of county-level smoking ban in place in 2010, taking one of the values "none", "partial", or "comprehensive".

Source
These data were collected from Census Quick Facts (no longer available as of 2020) and its accompanying pages. Smoking ban data were from a variety of sources.

In [97]:
!pip install pandas

In [98]:
!pip install plotly

In [99]:
!pip install scipy

In [100]:
!pip install statsmodels

In [102]:
# Imports all packages and loads county dataset from openintro

import numpy as np
import pandas as pd
import plotly.express as px
import requests
from io import StringIO
from scipy import stats
import statsmodels.formula.api as smf

url = "https://www.openintro.org/data/csv/county.csv"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
response = requests.get(url, headers=headers)
response.raise_for_status() # Raise an exception for bad status codes
df_county = pd.read_csv(StringIO(response.text))

In [103]:
df_county.head()

,name,state,pop2000,pop2010,pop2017,pop_change,poverty,homeownership,multi_unit,unemployment_rate,metro,median_edu,per_capita_income,median_hh_income,smoking_ban
0,Autauga County,Alabama,43671.0,54571,55504.0,1.48,13.7,77.5,7.2,3.86,yes,some_college,27841.70,55317.0,none
1,Baldwin County,Alabama,140415.0,182265,212628.0,9.19,11.8,76.7,22.6,3.99,yes,some_college,27779.85,52562.0,none
2,Barbour County,Alabama,29038.0,27457,25270.0,-6.22,27.2,68.0,11.1,5.90,no,hs_diploma,17891.73,33368.0,partial
3,Bibb County,Alabama,20826.0,22915,22668.0,0.73,15.2,82.9,6.6,4.39,yes,hs_diploma,20572.05,43404.0,none
4,Blount County,Alabama,51024.0,57322,58013.0,0.68,15.6,82.0,3.7,4.02,yes,hs_diploma,21367.39,47412.0,none


In [83]:
df_county = df_county.dropna()

In [84]:
df_county['poverty'].describe()

count    2560.000000
mean       16.281836
std         6.543054
min         2.400000
25%        11.600000
50%        15.600000
75%        19.800000
max        48.200000
Name: poverty, dtype: float64

In [85]:
df_county['median_hh_income'].describe()

count      2560.000000
mean      49047.173438
std       12942.529993
min       19264.000000
25%       40598.000000
50%       47132.000000
75%       55109.250000
max      129588.000000
Name: median_hh_income, dtype: float64

### **Scatterplot (without Regression line first)**

Just examining the scatterplot of the data, there is a clear negative relationship to see, meaning as the percentage of the population in poverty increases, our scatterplot shows that the average median household income would decrease as a result.

In [86]:
fig = px.scatter(
    df_county,
    x="poverty",
    y="median_hh_income",
    labels={"poverty":"Percent of population in poverty","median_hh_income":"Median household income ($)"}
)
fig.show()

### **Calculate m and b values to get the linear regression line equation**

Using ployfit, our m value is (-1494.5), meaning for every increase of 1 percentage of population in poverty the median household income decreases on average by 1494.50 dollars. The b value we get is approx 73380.39, meaning that according to our regression, if the percentage of population in poverty for a county is 0, we would expect the median household income to be 73380.39 dollars, although this is an impossible value that is not shown in the actual data.

In [87]:
# Fit linear regression line (y = mx + b)
m, b = np.polyfit(df_county["poverty"], df_county["median_hh_income"], 1)
print(m,b)
df_county["hh_income_pred"] = m * df_county["poverty"] + b      # hh_income_pred (household income prediction)

-1494.5010177378924 73380.39381673513


### **Scatterplot (with Regression line)**

Regression line means on the graph represents the "best fit" line that summarizes the relationship between poverty rates and median household incomes.
Shows the average predicted income for each poverty level. The line in the graph slopes DOWNWARD (negative relationship confirmed) with most points are close to the line = strong relationship. This line helps us make predictions for median household incomes with poverty rates not shown in the dataset.
The tightness of clustering shows how accurate our predictions are, or the approximate R^2 value, which is negative and decently high in this case, with a lot of points near the regression line.

In [101]:
fig = px.scatter(
    df_county,
    x="poverty",
    y="median_hh_income",
    labels={"poverty":"Percent of population in poverty","median_hh_income":"Median household income ($)"},
    title="Negative Relationship: Poverty vs Household Income"
)
fig.add_scatter(x=df_county["poverty"], y=df_county["hh_income_pred"], mode="lines",name='Regression line (prediction)')
fig.show()

### **OLS Regression Statictics**

Important Statictics: 

- Coefficient Values:
  The intercept (b = 73380.39) confirms our b value from our regression equation calculations, means when the poverty percentage is 0 the predicted income is     around $73,380.
  The poverty coefficient (m = -1494.50) means that for every 1% increase in poverty, the income decreases by around 1494. Negative value means the
  relationship is both inverse and negative.

- R-Squared:
  Value of 0.571 means that 57.1% of the variation in median household incomes is explained by poverty rates. Considered a good model fit for real-world data,    because more than half (50%) is explained. Shown in our scatterplot with points clustering mostly near the regression line.
  
- Confidence Interval:
  Poverty coefficient [-1544.741, -1444.261] means that we are 95% confident that the true slope is somewhere between -1544.741 and -1444.261. The range here     is narrow meaning that our estimate is precise. Also means that we're 95% sure that for every 1% poverty increase, income drops 
  between 1444.26 and 1544.74 dollars.

In [89]:
import statsmodels.formula.api as smf
model = smf.ols('median_hh_income ~ poverty',data=df_county).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       median_hh_income   R-squared:                       0.571
Model:                            OLS   Adj. R-squared:                  0.571
Method:                 Least Squares   F-statistic:                     3402.
Date:                Wed, 15 Jul 2026   Prob (F-statistic):               0.00
Time:                        22:43:20   Log-Likelihood:                -26788.
No. Observations:                2560   AIC:                         5.358e+04
Df Residuals:                    2558   BIC:                         5.359e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   7.338e+04    449.570    163.223      0.0

### **Pearson Correlation Coefficient (r)**

Correlation (r) of (-0.7555) is strong and negative with it being close to -1. As the poverty increases, income decreases predictably.

In [90]:
np.corrcoef(df_county['median_hh_income'],df_county['poverty'])[0,1]

-0.755540185183725

### **Summary of Exercise 1**

There is a Strong Negative relationship between the percent of population in poverty in each county of the United States and 
the median household income in each county of the United States.
According to our linear regression equation model, for every 1% increase in poverty, income decreases by around 1494 dollars.
We are 95% confident this slope is between 1444 and 1545.
This relationship is statistically significant and real (p < 0.001).
The model explains 57.1% of income variation.

# **EXERCISE 2: Identify Relationships between two variables**

### **Negative Relationship (From Exercise 1)**

Variables: Poverty Rate vs Median Household Income
Correlation: r = -0.7555 (Strong Negative)
As poverty rate increases, the income decrease by $1494.50 per 1%.

### **Postive Relationship (Population 2010 vs 2017)**

Using ployfit, our m value is (1.071), meaning for every increase of 1 person of the population in 2010 the 2017 population increases on average by 1.071 people. The b value we get is approx -877.02, meaning that according to our regression, if the 2010 population could be 0 poeple than the predicited 2017 population would be around -877.02, which is obviously impossible to have a negative population, which is why is it only a predicited value not showed in the data.

In [91]:
# Fit linear regression line (y = mx + b)
df_pos = df_county[['pop2010', 'pop2017']].dropna()
m, b = np.polyfit(df_pos["pop2010"], df_pos["pop2017"], 1)
print(m, b)
df_pos["pop2017_pred"] = m * df_pos["pop2010"] + b

1.071052367945043 -877.0234752828991


### **Scatterplot (with Regression line)**

The regression line shows an upward-sloping line (positive relationship) with both variables increasing together. 
There is also tight clustering of data with the points are very close to the line meaning the predicited values from the regression model and the acctual values are very similar. A nearly perfect fit, just looking at the graph R^2 is very close to 1.0, almost all variation is explained. This is likely just caused because population size is stable relatively stable over time, with large counties staying large, small staying small. The nearly perfect clustering shows this is a much stronger relationship than the poverty and income example above. Population 2010 is a very strong predictor of population 2017.

In [92]:
fig2 = px.scatter(
    df_pos,
    x="pop2010",
    y="pop2017",
    labels={"pop2010":"Population in 2010","pop2017":"Population in 2017"},
    title="Positive Relationship: Population 2010 vs 2017"
)
fig2.add_scatter(x=df_pos["pop2010"], y=df_pos["pop2017_pred"], mode="lines", name='Regression line')
fig2.show()

### **Pearson Correlation Coefficient (r)**

Correlation (r) of (0.998) is a very strong and positive relationship, confirming the graph above with regression line. Counties with large 2010 populations tend to have large 2017 populations. This makes sense with population size is stable over time again, large counties stay large, small counties stay small.

In [93]:
np.corrcoef(df_pos['pop2017'],df_pos['pop2010'])[0,1]

0.998849812823322

### **No Relationship (Homeownership Rates vs Unemployement Rates)**

Using ployfit, our m value is very small at around (-0.0171), meaning for every increase of homeownership rate, the unemployement rates drop by just -0.0171 on average. The b value we get is approx 5.856, meaning that according to our regression, if the homeownership could be 0 than the predicited unemployement rate would be around 5.856.

In [94]:
df_noCorr = df_county[['homeownership', 'unemployment_rate']].dropna()
m, b = np.polyfit(df_noCorr["homeownership"], df_noCorr["unemployment_rate"], 1)
print(m, b)
df_noCorr["unemployment_pred"] = m * df_noCorr["homeownership"] + b

-0.017144993384144137 5.856346212651594


### **Scatterplot (with Regression line)**

The regression line shows a nearly flat line with a slope around 0, meaning no strong relationship. There is a random scatter of points without any real pattern. Tells us that Homeownership rates tell us almost nothing about unemployment rates.

In [95]:
fig3 = px.scatter(
    df_noCorr,
    x="homeownership",
    y="unemployment_rate",
    title="No Relationship: Homeownership vs Unemployment"
)
fig3.add_scatter(x=df_noCorr["homeownership"], y=df_noCorr["unemployment_pred"], 
                 mode="lines", name='Regression line')
fig3.show()

### **Pearson Correlation Coefficient (r)**

Correlation (r) of (-0.077) is a extremely weak and negative relationship, confirming the graph above with regression line. Just shows us that according to our data these variables are independent. Someone can own a home and still be unemployed could maybe use savings or have it passed down. On the other hand, someone can easily rent and yet be employed.

In [96]:
np.corrcoef(df_noCorr['unemployment_rate'],df_noCorr['homeownership'])[0,1]

-0.07760505125517363